# Test BaselineCNN

This notebook is an interactive playground to test the `BaselineCNN` architecture initialized in our repository. 
We will verify that it correctly builds, accepts the `(Batch, 1, 128, 128)` shape, and outputs the expected `(Batch, 4)` logits. 

In [8]:
import sys
import torch
sys.path.insert(0, '../src')

from cochleogram_vit.models.baseline_cnn import BaselineCNN

# Setup parameters matching our cochleogram configuration
BATCH_SIZE = 4
IN_CHANNELS = 1
IMAGE_SIZE = 128
NUM_CLASSES = 4

print(f"Testing environment initialized!")
print(f"PyTorch version: {torch.__version__}")

Testing environment initialized!
PyTorch version: 2.5.1+cu121


## 1. Instantiate the Model
We verify that the model correctly instantiates without any missing layers or syntax errors.

In [9]:
# Initialize the model
model = BaselineCNN(in_channels=IN_CHANNELS, num_classes=NUM_CLASSES)

def count_parameters(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

print(model)
print("="*60)
print(f"Total trainable parameters: {count_parameters(model):,}")

BaselineCNN(
  (features): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU(inplace=True)
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (8): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU(inplace=True)
    (11): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (12): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): BatchNorm2d(256, eps=1e-05, momentum=0.1, 

## 2. Forward Pass Test (Dummy Batch)
We create a synthetic batch of random numbers matching the expected output of `CochleogramTransform`, and pass it through the model. 

If this step executes without shape mismatch errors, the model is fully compatible with our `trainer.py` engine!

In [3]:
# Create a dummy batch resembling a cochleogram tensor: (B, C, H, W)
dummy_input = torch.randn(BATCH_SIZE, IN_CHANNELS, IMAGE_SIZE, IMAGE_SIZE)
print(f"Input shape: {dummy_input.shape}")

# Set to evaluation mode to disable dropout during this precise verification step
model.eval()

# Forward pass
with torch.no_grad():
    logits = model(dummy_input)

print(f"Output shape: {logits.shape}")

# Verify expected output
assert logits.shape == (BATCH_SIZE, NUM_CLASSES), f"Output shape mismatch! Expected {(BATCH_SIZE, NUM_CLASSES)}, got {logits.shape}"
print("\nSuccess! ✨")
print("The BaselineCNN accepted the (B, 1, 128, 128) input and outputted (B, 4) logits.")
print("It is ready for the training pipeline.")

# Sample Logit Outputs
print("\nSample Logits (Raw scores):")
print(logits)

print("\nSample Probabilities (Softmaxed):")
print(torch.softmax(logits, dim=1))

Input shape: torch.Size([4, 1, 128, 128])
Output shape: torch.Size([4, 4])

Success! ✨
The BaselineCNN accepted the (B, 1, 128, 128) input and outputted (B, 4) logits.
It is ready for the training pipeline.

Sample Logits (Raw scores):
tensor([[-0.0202, -0.0792, -0.0131,  0.0429],
        [-0.0209, -0.0794, -0.0132,  0.0433],
        [-0.0203, -0.0794, -0.0128,  0.0429],
        [-0.0202, -0.0795, -0.0132,  0.0434]])

Sample Probabilities (Softmaxed):
tensor([[0.2491, 0.2348, 0.2508, 0.2653],
        [0.2489, 0.2348, 0.2509, 0.2654],
        [0.2490, 0.2348, 0.2509, 0.2653],
        [0.2491, 0.2347, 0.2508, 0.2654]])


In [ ]:
# ---------------------------------------------------------
# 3. 10-Fold Cross Validation Training (Patient-wise)
# ---------------------------------------------------------
# Implementing the exact procedure from the paper:
# - 10-fold patient-wise splitting to prevent data leakage.
# - Aggregating the predictions across all folds.
# ---------------------------------------------------------

import os
from sklearn.model_selection import GroupKFold
from torch.utils.data import DataLoader, Subset

from cochleogram_vit.utils.config import get_device, load_config, seed_everything
from cochleogram_vit.training.trainer import Trainer
import pandas as pd
import numpy as np
import torch

# Load configuration
cfg = load_config('../configs/default.yaml')

# Cast scientific notation properly
cfg["training"]["learning_rate"] = float(cfg["training"]["learning_rate"])
cfg["training"]["weight_decay"] = float(cfg["training"]["weight_decay"])

# Fix relative paths
cfg["data"]["raw_dir"] = f"../{cfg['data']['raw_dir']}"
cfg["data"]["processed_dir"] = f"../{cfg['data']['processed_dir']}"
cfg["logging"]["log_dir"] = f"../{cfg['logging']['log_dir']}"
cfg["logging"]["save_dir"] = f"../{cfg['logging']['save_dir']}"

# --- PAPER DEFINED HYPERPARAMETERS ---
cfg["training"]["epochs"] = 30
cfg["training"]["batch_size"] = 16
cfg["training"]["learning_rate"] = 1e-3
cfg["training"]["optimizer"] = "Adam"
cfg["training"]["num_workers"] = 0

seed_everything(cfg["training"]["seed"])
device = get_device(cfg["training"]["device"])
print(f"Training on device: {device}")

# Precomputed Dataset with Patient ID Extraction
class PrecomputedCochleogramDataset(torch.utils.data.Dataset):
    def __init__(self, metadata_csv: str):
        self.df = pd.read_csv(metadata_csv)
        self.df['npy_path'] = self.df['npy_path'].apply(lambda x: '../' + x)
        # Extract Patient ID (First three characters in ICBHI convention, eg '101_1b1...')
        self.df['patient_id'] = self.df['npy_path'].apply(lambda x: os.path.basename(x).split('_')[0])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        cg = np.load(row['npy_path'])
        tensor = torch.from_numpy(cg).unsqueeze(0)
        return tensor, int(row['label'])

metadata_csv = f"{cfg['data']['processed_dir']}/metadata.csv"
dataset = PrecomputedCochleogramDataset(metadata_csv)
print(f"Total precomputed cochleograms loaded: {len(dataset)}")
print(f"Unique patients found: {dataset.df['patient_id'].nunique()}")

# --- SET UP 10-FOLD PATIENT-WISE SPLIT ---
groups = dataset.df['patient_id'].values
gkf = GroupKFold(n_splits=10)

# Global lists to build the "Aggregated Confusion Matrix" at the end
global_targets = []
global_preds = []

print("\nStarting 10-Fold Patient-wise Cross Validation...")
print("This will train 10 separate models sequentially. Grab a coffee! ☕\n")

for fold, (train_idx, val_idx) in enumerate(gkf.split(dataset, groups=groups)):
    print(f"\n{'='*40}")
    print(f"                 FOLD {fold + 1}/10")
    print(f"{'='*40}")
    print(f"Train size: {len(train_idx)} | Val size: {len(val_idx)}")
    
    train_set = Subset(dataset, train_idx)
    val_set = Subset(dataset, val_idx)
    
    train_loader = DataLoader(train_set, batch_size=cfg["training"]["batch_size"], shuffle=True)
    val_loader = DataLoader(val_set, batch_size=cfg["training"]["batch_size"], shuffle=False)
    
    # Init a fresh model from scratch for each fold
    model = BaselineCNN(
        in_channels=cfg["model"].get("channels", 1),
        num_classes=cfg["model"]["num_classes"]
    ).to(device)
    
    trainer = Trainer(
        model=model, train_loader=train_loader, val_loader=val_loader,
        cfg=cfg, device=device
    )
    
    # Train the model for 30 epochs
    trainer.fit()
    
    # Evaluate Validation Set and Append to Globals
    model.eval()
    with torch.no_grad():
        for inputs, targets in val_loader:
            inputs = inputs.to(device)
            logits = model(inputs)
            preds = logits.argmax(dim=-1).cpu().numpy()
            
            global_preds.extend(preds)
            global_targets.extend(targets.numpy())

print("\n🏁 10-Fold CV Finished! The predictions are stored in memory.")
print("Run the next cell to build the Aggregated Confusion Matrix and compute the exact scores.")

In [18]:
# ---------------------------------------------------------
# 4. Aggregated ICBHI Metric Evaluation
# ---------------------------------------------------------
# Using the exact methodology from the paper:
# Creating an "Aggregated Confusion Matrix" from the 10 folds
# and calculating the official 4-class ICBHI Score.
# ---------------------------------------------------------

import numpy as np
from sklearn.metrics import confusion_matrix

print("Calculating the Aggregated Confusion Matrix (All 10 Folds combined)...")
n_classes = 4

# Create the global 4x4 matrix encompassing all predictions
cm_total = confusion_matrix(global_targets, global_preds, labels=list(range(n_classes)))

print("\n--- AGGREGATED 4x4 CONFUSION MATRIX ---")
print("Target rows vs Predict columns (0: Normal, 1: Crackle, 2: Wheeze, 3: Both)")
print(cm_total)

# --- EXACT ICBHI PAPER LOGIC (4-Class Scoring) ---
# For 4-class, TP is strictly the diagonal elements of the anomaly classes.
# A Crackle predicted as a Wheeze is NOT a True Positive!

TN = cm_total[0, 0]                                             # Normal correctly classified as Normal
TP = cm_total[1, 1] + cm_total[2, 2] + cm_total[3, 3]           # Adventitious correctly classified (Exact match)
FP = cm_total[0, 1:].sum()                                      # Normal wrongly flagged as any Adventitious class
FN = cm_total[1:, 0].sum()  

print("\n--- 4-CLASS EVALUATION METRICS ---")
print(f"True Negatives (TN) : {TN} (Healthy recognized as Healthy)")
print(f"True Positives (TP) : {TP} (Anomalies correctly classified into their EXACT class)")
print(f"False Positives (FP): {FP} (Healthy wrongly flagged as Anomaly)")
print(f"False Negatives (FN): {FN} (Anomalies missed or misclassified)")

# Extract explicit Sensitivity & Specificity ensuring we don't divide by absolute zero
# Sen = exact matches / total adventitious samples
# Spe = exact normal / total normal samples
Sen = TP / (TP + FN + 1e-8)
Spe = TN / (TN + FP + 1e-8)

# The overall ICBHI Score
Score = (Sen + Spe) / 2.0

print("\n--- FINAL COMPUTED METRICS vs PAPER ---")
print(f"Computed Sensitivity (Sen): {Sen * 100:.2f}%  | Target vs BaselineCNN: 52.7%")
print(f"Computed Specificity (Spe): {Spe * 100:.2f}%  | Target vs BaselineCNN: 68.8%")
print(f"Computed Score (Sco)      : {Score * 100:.2f}%  | Target vs BaselineCNN: 60.7%")

Calculating the Aggregated Confusion Matrix (All 10 Folds combined)...

--- AGGREGATED 4x4 CONFUSION MATRIX ---
Target rows vs Predict columns (0: Normal, 1: Crackle, 2: Wheeze, 3: Both)
[[2869  563  167   43]
 [1052  722   50   40]
 [ 364   68  365   89]
 [ 145   81  222   58]]

--- 4-CLASS EVALUATION METRICS ---
True Negatives (TN) : 2869 (Healthy recognized as Healthy)
True Positives (TP) : 1145 (Anomalies correctly classified into their EXACT class)
False Positives (FP): 773 (Healthy wrongly flagged as Anomaly)
False Negatives (FN): 1561 (Anomalies missed or misclassified)

--- FINAL COMPUTED METRICS vs PAPER ---
Computed Sensitivity (Sen): 42.31%  | Target vs BaselineCNN: 52.7%
Computed Specificity (Spe): 78.78%  | Target vs BaselineCNN: 68.8%
Computed Score (Sco)      : 60.54%  | Target vs BaselineCNN: 60.7%


In [19]:
# ---------------------------------------------------------
# 5. Wheeze & Crackle Specific Evaluation (Table 2 Format)
# ---------------------------------------------------------
# The paper evaluates Wheezes and Crackles independently.
# - Wheeze present in: Class 2 (Wheeze) and Class 3 (Both)
# - Crackle present in: Class 1 (Crackle) and Class 3 (Both)
# ---------------------------------------------------------

import numpy as np

targets = np.array(global_targets)
preds = np.array(global_preds)

def calc_binary_metrics(y_true, y_pred):
    """Calculate Sen, Spe, Score, Precision for binary classification"""
    TP = np.sum((y_true == 1) & (y_pred == 1))
    TN = np.sum((y_true == 0) & (y_pred == 0))
    FP = np.sum((y_true == 0) & (y_pred == 1))
    FN = np.sum((y_true == 1) & (y_pred == 0))
    
    sen = TP / (TP + FN + 1e-8)
    spe = TN / (TN + FP + 1e-8)
    sco = (sen + spe) / 2.0
    pre = TP / (TP + FP + 1e-8)
    
    return sen*100, spe*100, sco*100, pre*100

# --- Wheeze Binary Mapping ---
# 1 if class is 2 or 3, else 0
w_true = ((targets == 2) | (targets == 3)).astype(int)
w_pred = ((preds == 2) | (preds == 3)).astype(int)
w_sen, w_spe, w_sco, w_pre = calc_binary_metrics(w_true, w_pred)

# --- Crackle Binary Mapping ---
# 1 if class is 1 or 3, else 0
c_true = ((targets == 1) | (targets == 3)).astype(int)
c_pred = ((preds == 1) | (preds == 3)).astype(int)
c_sen, c_spe, c_sco, c_pre = calc_binary_metrics(c_true, c_pred)

print("\n" + "="*90)
print("BASELINE CNN (COCHLEOGRAM) - SPECIFIC METRICS TABLE (Paper Table 2 Format)")
print("="*90)
print(f"{'Metric':<20} | {'Wheezes':<35} | {'Crackles':<35}")
print("-"*90)
print(f"{'Sensitivity (Sen)':<20} | {w_sen:>6.2f}% (Paper: 67.7%) | {c_sen:>6.2f}% (Paper: 62.8%)")
print(f"{'Specificity (Spe)':<20} | {w_spe:>6.2f}% (Paper: 85.8%) | {c_spe:>6.2f}% (Paper: 75.8%)")
print(f"{'Score (Sco)':<20} | {w_sco:>6.2f}% (Paper: 76.7%) | {c_sco:>6.2f}% (Paper: 65.3%)")
print(f"{'Precision (Pre)':<20} | {w_pre:>6.2f}% (Paper: 50.3%) | {c_pre:>6.2f}% (Paper: 50.3%)")
print("="*90)


BASELINE CNN (COCHLEOGRAM) - SPECIFIC METRICS TABLE (Paper Table 2 Format)
Metric               | Wheezes                             | Crackles                           
------------------------------------------------------------------------------------------
Sensitivity (Sen)    |  52.73% (Paper: 67.7%) |  38.02% (Paper: 62.8%)
Specificity (Spe)    |  94.55% (Paper: 85.8%) |  83.15% (Paper: 75.8%)
Score (Sco)          |  73.64% (Paper: 76.7%) |  60.58% (Paper: 65.3%)
Precision (Pre)      |  70.99% (Paper: 50.3%) |  54.15% (Paper: 50.3%)
